# 01 - Exploratory Data Analysis

This notebook explores the raw e-commerce dataset (products, users, interactions) before any cleaning or modeling. For the full, reusable EDA pipeline (used by the Streamlit dashboard) see `src/eda.py` — this notebook calls the same functions so the analysis stays in one place.

In [ ]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from utils import RAW_DIR
sns.set_theme(style='whitegrid')
%matplotlib inline

## Load raw data

In [ ]:
products = pd.read_csv(RAW_DIR / 'products.csv')
users = pd.read_csv(RAW_DIR / 'users.csv')
interactions = pd.read_csv(RAW_DIR / 'interactions.csv', parse_dates=['timestamp'])
print(products.shape, users.shape, interactions.shape)
products.head()

In [ ]:
interactions.head()

## Basic data quality checks

Look for missing values, duplicates, and unrealistic values before trusting any chart drawn from this data.

In [ ]:
print('Missing values (products):')
print(products.isna().sum())
print('\nMissing values (interactions):')
print(interactions.isna().sum())
print('\nDuplicate interaction rows:', interactions.duplicated().sum())
print('Ratings outside [1,5]:', ((interactions['rating'] < 1) | (interactions['rating'] > 5)).sum())

## Run the full EDA chart suite

`src/eda.py` generates every chart required by the project spec (popular products, categories, rating distribution, interactions per user/product, category & price distribution, user activity, and the popularity long-tail) and saves them to `data/processed/eda_charts/`. We call it here so this notebook produces the same figures without duplicating plotting code.

In [ ]:
import data_preprocessing, feature_engineering, eda
data_preprocessing.run()
eda.run()

In [ ]:
from utils import PROCESSED_DIR
from PIL import Image
import matplotlib.pyplot as plt
charts = sorted((PROCESSED_DIR / 'eda_charts').glob('*.png'))
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, chart in zip(axes.flat, charts):
    ax.imshow(Image.open(chart))
    ax.axis('off')
    ax.set_title(chart.stem, fontsize=9)
plt.tight_layout()
plt.show()

## Key observations

- Product popularity follows a **long-tail** distribution: a small   set of products absorb a disproportionate share of interactions   (Zipf-like), which is why collaborative filtering has a lot of   signal to exploit for popular items but very little for the tail.
- Ratings are **skewed toward 4-5 stars**, consistent with the   well-documented positivity bias in real e-commerce review data.
- Interaction counts per user and per product are both **heavily   right-skewed** — most users/products have few interactions, a few   have very many. This directly motivates the cold-start handling   built into the hybrid recommender.